# Task 3 — HDF5 Sensor Data
**Course:** STQD6014 Data Science  
**File:** `sensors.h5` — 3 sensors × 3 readings × 1,000 observations

**Tasks:**
1. Simulate sensor readings and write to HDF5
2. Read Sensor_2 data and compute summary statistics
3. Print formatted summary output

In [ ]:
import numpy as np
import h5py

np.random.seed(123)
n_readings = 1000

# ── Simulate readings for 3 sensors ──────────────────────────────────────────
# Temperature : 20–30 °C
# Humidity    : 40–60 %
# Pressure    : 1000–1020 hPa

sensors = {
    'Sensor_1': {'location': 'Lab A', 'sensor_type': 'Type_X'},
    'Sensor_2': {'location': 'Lab B', 'sensor_type': 'Type_Y'},
    'Sensor_3': {'location': 'Lab C', 'sensor_type': 'Type_Z'},
}

data = {}
for name in sensors:
    data[name] = {
        'Temperature': 20 + 10 * np.random.rand(n_readings),
        'Humidity':    40 + 20 * np.random.rand(n_readings),
        'Pressure':  1000 + 20 * np.random.rand(n_readings),
    }

# ── Write HDF5 file ───────────────────────────────────────────────────────────
with h5py.File('data/sensors.h5', 'w') as f:
    for name, attrs in sensors.items():
        grp = f.create_group(name)
        grp.create_dataset('Temperature', data=data[name]['Temperature'])
        grp.create_dataset('Humidity',    data=data[name]['Humidity'])
        grp.create_dataset('Pressure',    data=data[name]['Pressure'])
        grp.attrs['location']    = attrs['location']
        grp.attrs['date']        = '2025-12-15'
        grp.attrs['sensor_type'] = attrs['sensor_type']

print("HDF5 file written: data/sensors.h5")

## Inspect HDF5 structure

In [ ]:
with h5py.File('data/sensors.h5', 'r') as f:
    print("Groups:", list(f.keys()))
    for name in f.keys():
        grp = f[name]
        print(f"\n{name}  (location={grp.attrs['location']}, type={grp.attrs['sensor_type']})")
        for ds in grp.keys():
            arr = grp[ds][:]
            print(f"  {ds:12s}: shape={arr.shape}, dtype={arr.dtype}")

## Read Sensor_2 and compute summary statistics

In [ ]:
with h5py.File('data/sensors.h5', 'r') as f:
    s2 = f['Sensor_2']
    temp2 = s2['Temperature'][:]
    hum2  = s2['Humidity'][:]
    pres2 = s2['Pressure'][:]
    loc   = s2.attrs['location']
    stype = s2.attrs['sensor_type']

print(f"Summary statistics for Sensor_2  ({stype} @ {loc})")
print("-" * 42)

for label, arr in [("Temperature (°C)", temp2),
                    ("Humidity    (%)",  hum2),
                    ("Pressure  (hPa)", pres2)]:
    print(f"\n{label}")
    print(f"  Mean : {np.mean(arr):.3f}")
    print(f"  Max  : {np.max(arr):.3f}")
    print(f"  Min  : {np.min(arr):.3f}")
    print(f"  Std  : {np.std(arr):.3f}")

## Compare all three sensors

In [ ]:
print(f"{'Sensor':<10}  {'Variable':<15}  {'Mean':>8}  {'Std':>7}  {'Min':>8}  {'Max':>8}")
print("-" * 65)

with h5py.File('data/sensors.h5', 'r') as f:
    for sname in ['Sensor_1', 'Sensor_2', 'Sensor_3']:
        grp = f[sname]
        for var in ['Temperature', 'Humidity', 'Pressure']:
            arr = grp[var][:]
            print(f"{sname:<10}  {var:<15}  {np.mean(arr):8.3f}  {np.std(arr):7.3f}"
                  f"  {np.min(arr):8.3f}  {np.max(arr):8.3f}")